# 01 · Factor Models: CAPM → FF3 → FF5

**Goal.** Take a set of standard test portfolios and explain their returns with three
nested asset-pricing models, adding factors as we go:

1. **CAPM** — one factor: the market (`Mkt-RF`)
2. **Fama-French 3** — adds size (`SMB`) and value (`HML`)
3. **Fama-French 5** — adds profitability (`RMW`) and investment (`CMA`)

**The thing to watch.** Each model produces an *alpha* (intercept) per portfolio —
the average return the factors *fail* to explain. If the added factors are real
sources of priced risk, the alphas should **shrink toward zero** as we move
CAPM → FF3 → FF5. That trend is the whole point of this notebook.

---

### Roadmap

| Step | What | Status |
|------|------|--------|
| **1** | Load factor data + test portfolios, inspect & sanity-check | ✅ this notebook |
| 2 | Build excess returns for the test portfolios | next |
| 3 | Run CAPM on every portfolio | next |
| 4 | Run FF3 | next |
| 5 | Run FF5 | next |
| 6 | Compare alphas across models (table, heatmap, t-stats, GRS) | next |


## Step 0 · Setup

We import the `get_french` helper from `src/data_loader.py`, which downloads a
Ken French dataset and caches it locally as parquet (under `data/`, gitignored).
All French returns are **in percent per month**.

In [2]:
import sys, os
sys.path.append(os.path.abspath(".."))   # so we can import from src/

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.data_loader import get_french

pd.set_option("display.float_format", lambda x: f"{x:.4f}")
plt.rcParams["figure.figsize"] = (10, 5)

/Users/Parimah/anaconda3/lib/python3.11/site-packages/pandas/core/computation/expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.4' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/Users/Parimah/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


## Step 1 · Load the data

We need two things:

1. **Factors** — we pull the **5-factor** file `F-F_Research_Data_5_Factors_2x3`,
   which contains `Mkt-RF, SMB, HML, RMW, CMA, RF` in one place. CAPM uses just
   `Mkt-RF`; FF3 uses `Mkt-RF, SMB, HML`; FF5 uses all five. Pulling one file keeps
   the risk-free rate `RF` consistent across all three models.

   > ⚠️ *Caveat:* the `SMB` in the 5-factor file is built slightly differently from
   > the `SMB` in the original 3-factor file (it averages across more sorts). For a
   > learning exercise this is the standard, clean choice — just know the FF3 here
   > uses the 5-factor-style `SMB`.

2. **Test portfolios** — the classic `25_Portfolios_5x5`: 25 portfolios formed on
   a 5×5 sort of **size** × **book-to-market**. These are the canonical left-hand-side
   assets for testing Fama-French models.

In [3]:
# Monthly sample. French data starts 1963-07 for these series.
START, END = "1963-07", "2024-12"

factors = get_french("F-F_Research_Data_5_Factors_2x3", start=START, end=END)
portfolios = get_french("25_Portfolios_5x5", start=START, end=END)

print("factors:   ", factors.shape, "->", list(factors.columns))
print("portfolios:", portfolios.shape)

factors:    (738, 6) -> ['Mkt-RF', 'SMB', 'HML', 'RMW', 'CMA', 'RF']
portfolios: (738, 25)


### Inspect the factors

What each column means (all are monthly returns, in %):

| Factor | Name | What it captures | Long − short |
|---|---|---|---|
| `Mkt-RF` | Market | Excess return of the whole market over the risk-free rate | Market − T-bill |
| `SMB` | Size | Small-cap stocks tend to beat large-cap | Small − Big |
| `HML` | Value | Cheap (high book/price) beats expensive (growth) | High − Low B/M |
| `RMW` | Profitability | Profitable firms beat unprofitable ones | Robust − Weak |
| `CMA` | Investment | Conservative (low-investment) firms beat aggressive ones | Conservative − Aggressive |
| `RF` | Risk-free | The risk-free rate (T-bill); not a factor — used to form excess returns | — |

In [4]:
factors.head()

,Mkt-RF,SMB,HML,RMW,CMA,RF
Date,,,,,,
1963-07-31 23:59:59.999999,-0.3900,-0.4800,-0.8100,0.6400,-1.1500,0.2700
1963-08-31 23:59:59.999999,5.0800,-0.8000,1.7000,0.4000,-0.3800,0.2500
1963-09-30 23:59:59.999999,-1.5700,-0.4300,0.0000,-0.7800,0.1500,0.2700
1963-10-31 23:59:59.999999,2.5400,-1.3400,-0.0400,2.7900,-2.2500,0.2900
1963-11-30 23:59:59.999999,-0.8600,-0.8500,1.7300,-0.4300,2.2700,0.2700


In [5]:
factors.describe().T[["mean", "std", "min", "max"]]

,mean,std,min,max
Mkt-RF,0.5867,4.4760,-23.1900,16.1000
SMB,0.1930,3.0361,-15.5400,18.4600
HML,0.2819,2.9714,-13.8300,12.8600
RMW,0.2810,2.2172,-18.9500,13.0500
CMA,0.2509,2.0574,-7.0800,9.0100
RF,0.3637,0.2636,0.0000,1.3500


### Inspect the test portfolios

Columns are labeled `SIZE B/M`, e.g. `SMALL LoBM` (small-cap, low book-to-market /
growth) through `BIG HiBM` (large-cap, high book-to-market / value).

The 25 columns come in a fixed **row-major order**: outer loop is **size** (ME1→ME5,
small→big), inner loop is **book-to-market** (BM1→BM5, growth→value). Laid out as the
5×5 grid:

| Size \\ B/M | BM1 (Growth) | BM2 | BM3 | BM4 | BM5 (Value) |
|---|---|---|---|---|---|
| **ME1 (Small)** | `SMALL LoBM` | `ME1 BM2` | `ME1 BM3` | `ME1 BM4` | `SMALL HiBM` |
| **ME2** | `ME2 BM1` | `ME2 BM2` | `ME2 BM3` | `ME2 BM4` | `ME2 BM5` |
| **ME3** | `ME3 BM1` | `ME3 BM2` | `ME3 BM3` | `ME3 BM4` | `ME3 BM5` |
| **ME4** | `ME4 BM1` | `ME4 BM2` | `ME4 BM3` | `ME4 BM4` | `ME4 BM5` |
| **ME5 (Big)** | `BIG LoBM` | `ME5 BM2` | `ME5 BM3` | `ME5 BM4` | `BIG HiBM` |

The four corners are the extremes: `SMALL LoBM` (small growth), `SMALL HiBM`
(small value), `BIG LoBM` (large growth), `BIG HiBM` (large value).

In [6]:
print(list(portfolios.columns))
portfolios.head()

['SMALL LoBM', 'ME1 BM2', 'ME1 BM3', 'ME1 BM4', 'SMALL HiBM', 'ME2 BM1', 'ME2 BM2', 'ME2 BM3', 'ME2 BM4', 'ME2 BM5', 'ME3 BM1', 'ME3 BM2', 'ME3 BM3', 'ME3 BM4', 'ME3 BM5', 'ME4 BM1', 'ME4 BM2', 'ME4 BM3', 'ME4 BM4', 'ME4 BM5', 'BIG LoBM', 'ME5 BM2', 'ME5 BM3', 'ME5 BM4', 'BIG HiBM']


,SMALL LoBM,ME1 BM2,ME1 BM3,ME1 BM4,SMALL HiBM,ME2 BM1,ME2 BM2,ME2 BM3,ME2 BM4,ME2 BM5,...,ME4 BM1,ME4 BM2,ME4 BM3,ME4 BM4,ME4 BM5,BIG LoBM,ME5 BM2,ME5 BM3,ME5 BM4,BIG HiBM
Date,,,,,,,,,,,,,,,,,,,,,
1963-07-31 23:59:59.999999,1.1287,-0.3632,0.7223,-0.0413,-1.2447,-1.8076,0.1929,-1.0149,-1.9749,-1.1880,...,-0.9115,-1.7733,-1.9168,-1.5745,-1.8574,0.1391,0.4839,1.1360,-0.4285,-1.1045
1963-08-31 23:59:59.999999,4.2396,1.3730,1.4917,2.5068,4.6644,5.5703,4.5220,4.4450,4.4662,8.2451,...,5.5754,4.7469,6.2516,7.6941,5.3456,5.7823,4.2633,4.6341,8.1704,6.3984
1963-09-30 23:59:59.999999,-1.7343,0.6204,-1.0007,-1.5215,-0.3584,-4.0525,-1.5072,-0.8638,-1.4935,-2.9243,...,-2.6644,-2.1463,-1.7882,-3.9641,-2.0002,-1.3752,-0.8081,-0.8497,-0.1912,-3.5033
1963-10-31 23:59:59.999999,0.3778,-0.7329,1.3066,0.1904,2.3711,1.1926,4.2411,2.3526,2.3058,3.9314,...,-0.2415,0.6990,2.5214,4.8524,0.6138,5.3261,1.7420,-0.3354,2.4176,0.4702
1963-11-30 23:59:59.999999,-3.3319,-3.8436,-1.7893,-1.0535,-1.1077,-4.2596,-1.7484,-0.7845,-0.0554,-0.1150,...,-0.9083,-0.6311,-0.7516,1.3596,3.5407,-1.2669,1.0080,-1.6914,-2.1316,1.3496


### Sanity checks

In [7]:
# 1) Indexes align on the same monthly dates
assert factors.index.equals(portfolios.index), "date indexes do not match!"

# 2) No missing values (French uses -99.99 for missing; flag if present)
print("factors  has -99.99 sentinels:", (factors == -99.99).any().any())
print("ports    has -99.99 sentinels:", (portfolios == -99.99).any().any())
print("factors  NaNs:", int(factors.isna().sum().sum()))
print("ports    NaNs:", int(portfolios.isna().sum().sum()))

# 3) Date range
print("range:", factors.index.min().date(), "->", factors.index.max().date(),
      f"({len(factors)} months)")

factors  has -99.99 sentinels: False
ports    has -99.99 sentinels: False
factors  NaNs: 0
ports    NaNs: 0
range: 1963-07-31 -> 2024-12-31 (738 months)


**Quick read of the factors** (means are % per month — annualize by ×12):

- `Mkt-RF` ≈ the equity risk premium
- `RF` is the risk-free rate; everything else is already a *long-short factor return*
  (zero-cost portfolio), so those don't need RF subtracted — only the **test portfolios** do.

That's exactly Step 2.

---

---

### ✅ Step 1 done

Aligned monthly factors and 25 test portfolios, sanity-checked. On to Step 2.

## Step 2 · Excess returns & factor sets

Two small but conceptually important pieces of prep before any regression.

**(a) Excess returns for the test portfolios.** Asset-pricing regressions explain
*excess* returns — return above the risk-free rate. So we subtract `RF` from each of
the 25 portfolios:
$$R^{e}_{i,t} = R_{i,t} - R_{f,t}$$
We do **not** subtract `RF` from the factors: `Mkt-RF` is already an excess return, and
`SMB, HML, RMW, CMA` are zero-cost long-short spreads (a dollar long funds a dollar
short, so there's no risk-free leg to net out).

**(b) Factor sets.** The three nested models are just growing column lists of the same
factor table:

| Model | Factors |
|---|---|
| CAPM | `Mkt-RF` |
| FF3  | `Mkt-RF, SMB, HML` |
| FF5  | `Mkt-RF, SMB, HML, RMW, CMA` |

In [8]:
# (a) Excess returns: subtract RF from every portfolio column
rf = factors["RF"]
excess = portfolios.sub(rf, axis=0)

print("excess shape:", excess.shape)
excess.head()

excess shape: (738, 25)


,SMALL LoBM,ME1 BM2,ME1 BM3,ME1 BM4,SMALL HiBM,ME2 BM1,ME2 BM2,ME2 BM3,ME2 BM4,ME2 BM5,...,ME4 BM1,ME4 BM2,ME4 BM3,ME4 BM4,ME4 BM5,BIG LoBM,ME5 BM2,ME5 BM3,ME5 BM4,BIG HiBM
Date,,,,,,,,,,,,,,,,,,,,,
1963-07-31 23:59:59.999999,0.8587,-0.6332,0.4523,-0.3113,-1.5147,-2.0776,-0.0771,-1.2849,-2.2449,-1.4580,...,-1.1815,-2.0433,-2.1868,-1.8445,-2.1274,-0.1309,0.2139,0.8660,-0.6985,-1.3745
1963-08-31 23:59:59.999999,3.9896,1.1230,1.2417,2.2568,4.4144,5.3203,4.2720,4.1950,4.2162,7.9951,...,5.3254,4.4969,6.0016,7.4441,5.0956,5.5323,4.0133,4.3841,7.9204,6.1484
1963-09-30 23:59:59.999999,-2.0043,0.3504,-1.2707,-1.7915,-0.6284,-4.3225,-1.7772,-1.1338,-1.7635,-3.1943,...,-2.9344,-2.4163,-2.0582,-4.2341,-2.2702,-1.6452,-1.0781,-1.1197,-0.4612,-3.7733
1963-10-31 23:59:59.999999,0.0878,-1.0229,1.0166,-0.0996,2.0811,0.9026,3.9511,2.0626,2.0158,3.6414,...,-0.5315,0.4090,2.2314,4.5624,0.3238,5.0361,1.4520,-0.6254,2.1276,0.1802
1963-11-30 23:59:59.999999,-3.6019,-4.1136,-2.0593,-1.3235,-1.3777,-4.5296,-2.0184,-1.0545,-0.3254,-0.3850,...,-1.1783,-0.9011,-1.0216,1.0896,3.2707,-1.5369,0.7380,-1.9614,-2.4016,1.0796


In [9]:
# Sanity: raw vs excess means for a few portfolios (RF > 0, so excess mean < raw mean)
check = pd.DataFrame({
    "raw_mean":    portfolios.mean(),
    "excess_mean": excess.mean(),
    "diff(=mean RF)": portfolios.mean() - excess.mean(),
}).head()
print(f"mean RF over sample = {rf.mean():.4f}% / month")
check

mean RF over sample = 0.3637% / month


,raw_mean,excess_mean,diff(=mean RF)
SMALL LoBM,0.6532,0.2895,0.3637
ME1 BM2,1.1206,0.7568,0.3637
ME1 BM3,1.1277,0.7639,0.3637
ME1 BM4,1.3130,0.9492,0.3637
SMALL HiBM,1.4614,1.0976,0.3637


In [10]:
# (b) Define the three nested model specifications
MODELS = {
    "CAPM": ["Mkt-RF"],
    "FF3":  ["Mkt-RF", "SMB", "HML"],
    "FF5":  ["Mkt-RF", "SMB", "HML", "RMW", "CMA"],
}

# Confirm every factor we reference actually exists in the factors table
for name, cols in MODELS.items():
    missing = [c for c in cols if c not in factors.columns]
    assert not missing, f"{name}: missing {missing}"
    print(f"{name:5s}: {cols}")

CAPM : ['Mkt-RF']
FF3  : ['Mkt-RF', 'SMB', 'HML']
FF5  : ['Mkt-RF', 'SMB', 'HML', 'RMW', 'CMA']


### ✅ Step 2 done

We have `excess` (738 × 25 excess returns) and `MODELS` (the three factor specs).

**Next (Step 3):** run **CAPM** — regress each of the 25 portfolios' excess returns on
`Mkt-RF` alone, and collect the alphas (intercepts) and their t-stats. That's our
baseline before adding factors.

## Step 3 · CAPM — the baseline

The CAPM says one factor — the market — explains everything, so for each portfolio $i$:
$$R^{e}_{i,t} = \alpha_i + \beta_i\,(\text{Mkt-RF})_t + \varepsilon_{i,t}$$

We run this time-series regression for **each of the 25 portfolios** and read off:

- **$\alpha_i$** — average monthly return the market *fails* to explain. CAPM's claim is $\alpha_i = 0$ for all $i$.
- **$t(\alpha_i)$** — is that alpha statistically distinguishable from zero? $|t| > 1.96$ ≈ significant at 5%.
- **$\beta_i$** — market exposure; **$R^2$** — how much of the variance the market explains.

A reusable `run_model` helper does the regression for *any* factor set, so Steps 4 and 5
(FF3, FF5) just call it again with more columns.

In [11]:
import statsmodels.api as sm

def run_model(excess_df, factors_df, factor_cols):
    """OLS time-series regression of each portfolio's excess return on `factor_cols`.

    Returns a DataFrame (one row per portfolio) with alpha, t(alpha), R², and betas.
    """
    rows = {}
    X = sm.add_constant(factors_df[factor_cols])      # design matrix shared across portfolios
    for col in excess_df.columns:
        res = sm.OLS(excess_df[col], X).fit()
        rows[col] = {
            "alpha":   res.params["const"],
            "t_alpha": res.tvalues["const"],
            "R2":      res.rsquared,
            **{f"b_{f}": res.params[f] for f in factor_cols},
        }
    return pd.DataFrame(rows).T

def to_grid(series):
    """Reshape a 25-vector (row-major size×B/M) into the labeled 5×5 grid."""
    return pd.DataFrame(
        series.values.reshape(5, 5).astype(float),
        index=["ME1 (Small)", "ME2", "ME3", "ME4", "ME5 (Big)"],
        columns=["BM1 (Growth)", "BM2", "BM3", "BM4", "BM5 (Value)"],
    )


RESULTS = {}   # model name -> per-portfolio DataFrame
ALPHAS  = {}   # model name -> alpha Series (for the Step 6 comparison)

In [12]:
RESULTS["CAPM"] = run_model(excess, factors, MODELS["CAPM"])
ALPHAS["CAPM"]  = RESULTS["CAPM"]["alpha"]

### CAPM alphas as a 5×5 grid (% per month)

In [13]:
print("Alpha grid:")
display(to_grid(ALPHAS["CAPM"]).round(3))
print("\nt(alpha) grid:")
display(to_grid(RESULTS["CAPM"]["t_alpha"]).round(2))

Alpha grid:


,BM1 (Growth),BM2,BM3,BM4,BM5 (Value)
ME1 (Small),-0.5370,0.0400,0.1110,0.3390,0.4700
ME2,-0.2760,0.0860,0.2210,0.2890,0.3150
ME3,-0.2340,0.1280,0.1570,0.2770,0.3420
ME4,-0.0530,0.0180,0.1240,0.2730,0.2170
ME5 (Big),0.0280,0.0220,0.0640,0.0140,0.1120



t(alpha) grid:


,BM1 (Growth),BM2,BM3,BM4,BM5 (Value)
ME1 (Small),-2.9600,0.2500,0.8500,2.5600,3.0800
ME2,-2.0000,0.7700,2.1300,2.7100,2.3800
ME3,-2.1200,1.4800,1.8400,2.9000,2.7100
ME4,-0.6300,0.2700,1.5600,3.0300,1.8100
ME5 (Big),0.4700,0.3800,0.8700,0.1500,0.8700


### CAPM summary

In [14]:
def summarize(name):
    r = RESULTS[name]
    n_sig = int((r["t_alpha"].abs() > 1.96).sum())
    return {
        "model": name,
        "mean |alpha|": r["alpha"].abs().mean(),
        "max |alpha|":  r["alpha"].abs().max(),
        "# |t|>1.96 (of 25)": n_sig,
        "mean R2": r["R2"].mean(),
    }

pd.DataFrame([summarize("CAPM")]).set_index("model").round(4)

,mean |alpha|,max |alpha|,# |t|>1.96 (of 25),mean R2
model,,,,
CAPM,0.1899,0.5371,11,0.7388


### Reading the baseline

Look at the alpha grid, especially down each column and across the small-cap row:

- CAPM leaves **economically large, statistically significant alphas** on many of the 25
  portfolios — that's the classic finding that the market factor alone *fails* to price
  the size/value sorts.
- The pattern is structured, not random: small-cap and high-B/M (value) corners tend to
  carry the biggest positive alphas. That structure is precisely what `SMB` and `HML` are
  built to absorb.

So we have our baseline. **Next (Step 4):** add `SMB` and `HML` (FF3) and watch these
alphas — and the count of significant ones — shrink.

# FF3

In [17]:
RESULTS["FF3"] = run_model(excess, factors, MODELS["FF3"])
ALPHAS["FF3"]  = RESULTS["FF3"]["alpha"]

## FF3 results in grid format

In [18]:
print("Alpha grid:")
display(to_grid(ALPHAS["FF3"]).round(3))
print("\nt(alpha) grid:")
display(to_grid(RESULTS["FF3"]["t_alpha"]).round(2))

Alpha grid:


,BM1 (Growth),BM2,BM3,BM4,BM5 (Value)
ME1 (Small),-0.4800,0.0020,-0.0230,0.1390,0.1920
ME2,-0.1780,0.0240,0.0660,0.0660,0.0050
ME3,-0.1170,0.0610,-0.0050,0.0430,0.0200
ME4,0.0700,-0.0550,-0.0370,0.0620,-0.0940
ME5 (Big),0.1620,-0.0030,-0.0380,-0.2200,-0.1870



t(alpha) grid:


,BM1 (Growth),BM2,BM3,BM4,BM5 (Value)
ME1 (Small),-5.2300,0.0200,-0.4500,2.8800,2.6000
ME2,-2.9300,0.4900,1.2500,1.4200,0.1100
ME3,-2.0500,1.1000,-0.0800,0.7900,0.2900
ME4,1.2100,-0.9100,-0.5900,0.9800,-1.2500
ME5 (Big),3.9600,-0.0600,-0.6100,-3.7600,-2.0200


## FF3 Summary

In [20]:
pd.DataFrame([summarize("FF3")]).set_index("model").round(4)

,mean |alpha|,max |alpha|,# |t|>1.96 (of 25),mean R2
model,,,,
FF3,0.0939,0.4796,8,0.9149


# FF5

In [22]:
RESULTS["FF5"] = run_model(excess, factors, MODELS["FF5"])
ALPHAS["FF5"]  = RESULTS["FF5"]["alpha"]


In [23]:
print("Alpha grid:")
display(to_grid(ALPHAS["FF5"]).round(3))
print("\nt(alpha) grid:")
display(to_grid(RESULTS["FF5"]["t_alpha"]).round(2))

Alpha grid:


,BM1 (Growth),BM2,BM3,BM4,BM5 (Value)
ME1 (Small),-0.2820,0.1420,0.0020,0.1530,0.1450
ME2,-0.0810,0.0160,0.0000,0.0290,-0.0110
ME3,-0.0240,0.0110,-0.0770,-0.0080,-0.0360
ME4,0.1360,-0.1450,-0.1280,0.0410,-0.0830
ME5 (Big),0.1150,-0.0920,-0.0810,-0.2210,-0.0320



t(alpha) grid:


,BM1 (Growth),BM2,BM3,BM4,BM5 (Value)
ME1 (Small),-3.3100,2.1300,0.0400,3.0900,1.9500
ME2,-1.3400,0.3200,0.0100,0.6200,-0.2200
ME3,-0.4200,0.1900,-1.3600,-0.1600,-0.5200
ME4,2.3400,-2.3900,-2.0600,0.6200,-1.0700
ME5 (Big),2.9300,-1.7100,-1.2600,-3.6800,-0.3500


In [24]:
pd.DataFrame([summarize("FF5")]).set_index("model").round(4)

,mean |alpha|,max |alpha|,# |t|>1.96 (of 25),mean R2
model,,,,
FF5,0.0835,0.2822,8,0.9195
